In [0]:
source_path='/Volumes/finance_ws/source/fraud_watchlist/source_data/'

In [0]:
dbutils.fs.ls(source_path)

In [0]:
input_stream=(spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format", "json")
              .option("cloudFiles.schemaLocation", "/Volumes/finance_ws/source/fraud_watchlist/source_data/")
              .option("cloudFiles.inferColumnTypes","true")
              .load(source_path)
)

In [0]:
from pyspark.sql import functions as F
tranformed_df=input_stream.select(
"*",
F.col("_metadata.file_path").alias("file_path"),
F.current_timestamp().alias("ingestion_timestamp")
)

In [0]:
streaming_query=(tranformed_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation", "/Volumes/finance_ws/source/fraud_watchlist/checkpointlocation/")
.trigger(availableNow=True)
.toTable("finance_ws.bronze.fraud_watchlist_batch_test")
)

In [0]:
# COMMAND ----------

# MAGIC %sql
# MAGIC select * from finguard.bronze.fraud_watchlist_batch_test

# COMMAND ----------

# MAGIC %sql
# MAGIC select * from finguard.bronze.fraud_watchlist